<a href="https://colab.research.google.com/github/fleurclochette/Scientific-Project-SS26/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import subprocess
import numpy as np
import pandas as pd
import yfinance as yf
from google.colab import userdata

# Securely fetching credentials from Colab Secrets
GH_TOKEN = userdata.get('GITHUB_TOKEN')
GH_USER = userdata.get('GITHUB_USER')
GH_EMAIL = "tushar.latiyan@st.ovgu.de"
REPO_URL = f"https://{GH_TOKEN}@github.com/fleurclochette/Scientific-Project-SS26.git"

# The full 23-asset universe
tickers = [
    '^GSPC', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY',
    '^AXJO', '^DJI', '^FCHI', '^FTSE', '^GDAXI', '^GSPTSE', '^HSI',
    '^IXIC', '^KS11', '^N225', '^RUT', '^STOXX50E', '^TWII'
]

print("Pulling market data from Jan 2000 through July 2026...")

# Added auto_adjust=False to stop yfinance from hiding the Adj Close column
df_raw = yf.download(tickers, start='2000-01-01', end='2026-08-01', auto_adjust=False)['Adj Close']

# Forward filling to sync global trading holidays, then drop the starting NaNs
prices = df_raw.ffill().dropna()

# Calculating daily log returns for the EGARCH/Bai-Perron models
log_returns = np.log(prices / prices.shift(1)).dropna()

# Saving it locally
os.makedirs('data/raw', exist_ok=True)
csv_path = 'data/raw/master_23_assets_returns_2000_2026.csv'
log_returns.to_csv(csv_path)

print(f"Done. Saved {len(log_returns)} rows from {log_returns.index.min().date()} to {log_returns.index.max().date()}.")

print("Pushing securely to the main branch...")
cmds = [
    "rm -rf temp_repo",
    f"git clone {REPO_URL} temp_repo",
    "mkdir -p temp_repo/data/raw",
    f"cp {csv_path} temp_repo/data/raw/",
    f'cd temp_repo && git config user.email "{GH_EMAIL}" && git config user.name "{GH_USER}"',
    'cd temp_repo && git add data/raw/master_23_assets_returns_2000_2026.csv',
    'cd temp_repo && git commit -m "chore: add unified 23-asset master CSV for 2000-2026"',
    'cd temp_repo && git push origin main',
    "rm -rf temp_repo"
]

for cmd in cmds:
    subprocess.run(cmd, shell=True, check=True)

print("Git push complete. The master file is ready for Karin.")

[                       0%                       ]

Pulling market data from Jan 2000 through July 2026...


[*********************100%***********************]  23 of 23 completed


Done. Saved 5031 rows from 2007-04-02 to 2026-07-31.
Pushing securely to the main branch...
Git push complete. The master file is ready for Karin.
